# Noise Dataset Preprocessing (STD1-028K, 96 kHz)

This notebook preprocesses (pure environmental / structural noise recordings)
acquired with a STD1-028K piezoelectric sensor and Focusrite Scarlett 2i2 (96 kHz).

The generated noise samples are:
- Time-aligned with click samples
- Band-limited to the informative frequency range
- Converted into Mel-spectrograms
- Saved as `.npy` files for CNN training

In [ ]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
# ==================================================
# Hardware-adapted parameters
# STD1-028K + Piezo Lab Amplifier + Scarlett 2i2
# ==================================================

SR = 96000                     # Sampling rate (Hz)

# Time parameters
CLIP_DURATION_MS = 20          # Noise clip length (must match click samples)
CLIP_STEP_MS = 10              # Overlap step (50%)

# Frequency parameters (piezo vibration domain)
F_MIN = 2000                   # Hz (remove low-frequency structure noise)
F_MAX = 40000                  # Hz (piezo click / noise content)

# Spectrogram parameters
N_MELS = 32                    # CNN input height
N_FFT = 4096                   # FFT size for 96 kHz
HOP_LENGTH = 512

# dB scaling
DB_FLOOR = -80                 # Lower dB limit


# ==================================================
# Path configuration
# ==================================================

# Folder containing pure noise recordings (.wav)
NOISE_WAV_DIR = os.path.expanduser(
    "~/Desktop/PA/sound/noise_recordings"
)

# Output dataset folder (noise samples)
OUTPUT_DIR = os.path.expanduser(
    "~/Desktop/PA/01_Dataset/01_audioDatasets/07_Noise_Samples_STD1"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Noise input directory:")
print(NOISE_WAV_DIR)
print("\nOutput dataset directory:")
print(OUTPUT_DIR)


In [ ]:
def compute_mel_spectrogram(x, sr):
    """
    Compute log-scaled Mel-spectrogram for one noise clip.
    """
    mel = librosa.feature.melspectrogram(
        y=x,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
        power=2.0
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = np.clip(mel_db, DB_FLOOR, 0)

    return mel_db


In [ ]:
clip_len = int(CLIP_DURATION_MS * 1e-3 * SR)
clip_step = int(CLIP_STEP_MS * 1e-3 * SR)

sample_index = 0

wav_files = [
    f for f in os.listdir(NOISE_WAV_DIR)
    if f.lower().endswith(".wav")
]

print(f"Found {len(wav_files)} noise recordings")

for wav_name in tqdm(wav_files):
    wav_path = os.path.join(NOISE_WAV_DIR, wav_name)

    x, sr = librosa.load(wav_path, sr=SR, mono=True)

    if len(x) < clip_len:
        continue

    for start in range(0, len(x) - clip_len, clip_step):
        clip = x[start:start + clip_len]

        # Skip silent or invalid segments
        if np.max(np.abs(clip)) < 1e-6:
            continue

        mel_db = compute_mel_spectrogram(clip, sr)

        out_path = os.path.join(
            OUTPUT_DIR,
            f"noise_{sample_index:06d}.npy"
        )

        np.save(out_path, mel_db)
        sample_index += 1

print(f"\nTotal noise samples generated: {sample_index}")


In [ ]:
# Load one random noise sample for inspection
files = os.listdir(OUTPUT_DIR)
example = np.load(os.path.join(OUTPUT_DIR, files[0]))

plt.figure(figsize=(4, 4))
librosa.display.specshow(
    example,
    sr=SR,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="mel",
    fmin=F_MIN,
    fmax=F_MAX,
    cmap="inferno"
)
plt.title("Noise Sample (STD1-028K)")
plt.colorbar(label="dB")
plt.tight_layout()
plt.show()
